# AeroPure — Week 1: Problem Definition & Exploratory Data Analysis
**Project Tagline**: *"Tell a city when tomorrow's air turns dangerous."*

### 1. Problem Overview
Air pollution poses an acute public health threat, yet citizens usually learn that air is toxic only after they have already inhaled it. 
AeroPure builds an intelligent next-day forecasting system. In Week 1, we conduct comprehensive exploratory data analysis (EDA) on the observational dataset (`data/AirQuality.csv`).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.preprocessing import load_raw_archive1, inspect_dataset

DATA_PATH = os.path.join("..", "data", "AirQuality.csv")
if not os.path.exists(DATA_PATH):
    print("Dataset not found. Please verify data/AirQuality.csv exists.")
else:
    df_raw = load_raw_archive1(DATA_PATH)
    print(f"Dataset successfully loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
    display(df_raw.head())


### 2. Dataset Dimensions, Types, and Sentinel (-200) Inspection
Let us inspect the dataset schema, data types, and check for missing/sentinel values.


In [ ]:
if "df_raw" in locals():
    stats = inspect_dataset(df_raw)
    print(f"Rows: {stats['shape'][0]}, Columns: {stats['shape'][1]}")
    print(f"Duplicate Rows: {stats['duplicate_rows']}")
    print("\nSentinel (-200) Counts per sensor:")
    for col, count in stats['sentinel_minus_200_counts'].items():
        print(f"  {col:15s}: {count}")


### 3. Summary Statistics & Contaminant Distributions
Visualizing the distributions of gaseous criteria contaminants and sensor responses.


In [ ]:
if "df_raw" in locals():
    from src.preprocessing import clean_dataset
    df_clean, _ = clean_dataset(df_raw)
    plot_cols = [c for c in ["CO(GT)", "NO2(GT)", "C6H6(GT)", "NOx(GT)", "T", "RH"] if c in df_clean.columns]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=120)
    axes = axes.flatten()
    for i, col in enumerate(plot_cols):
        sns.histplot(df_clean[col].dropna(), kde=True, ax=axes[i], color="#1f77b4")
        axes[i].set_title(f"{col} Distribution")
    plt.tight_layout()
    plt.show()


### 4. Correlation Analysis
Analyzing the correlation between atmospheric contaminants and meteorological factors (temperature, relative humidity).


In [ ]:
if "df_clean" in locals():
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    corr = df_clean[num_cols].corr()
    plt.figure(figsize=(11, 9), dpi=120)
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("AeroPure: Multi-Pollutant & Meteorological Correlation Matrix")
    plt.show()


### 5. Temporal Trends & Diurnal Cycles
Checking temporal progression and hourly variations to observe rush-hour peaks and atmospheric stagnation.


In [ ]:
if "df_clean" in locals():
    df_clean['hour'] = df_clean['datetime'].dt.hour
    hourly_means = df_clean.groupby('hour')[plot_cols[:4]].mean()
    plt.figure(figsize=(12, 5), dpi=120)
    for p in plot_cols[:4]:
        plt.plot(hourly_means.index, hourly_means[p], marker="o", label=p)
    plt.xlabel("Hour of Day (0–23)")
    plt.ylabel("Mean Level")
    plt.title("Diurnal Variation of Air Contaminants Across the Day")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()


### Week 1 Summary
- Cleaned European format data and mapped -200 sentinels to NaN.
- Dropped NMHC(GT) due to 90.2% missingness, preserving 12 high-quality sensor and meteorological channels.
- Identified prominent diurnal peaks corresponding to urban traffic and morning/evening boundary layer collapse.
